# SFT: Ministral-3-3B → web computer-use agent

**Runtime: A100 40GB** (Colab Pro). Runtime → Change runtime type → A100.

Trains `mistralai/Ministral-3-3B-Base-2512` (Apache 2.0, 3.4B LM + 0.4B Pixtral ViT) on the
web-only subset of `xlangai/aguvis-stage2`. Produces a **merged full SFT model**, not just an adapter.

| | |
|---|---|
| Data | mind2web (7,591) + guiact-web-single + miniwob — ~2.5 GB |
| Method | LoRA on the language model + trainable multimodal projector, ViT frozen |
| Budget | ~2–3 h on A100, hard wall-clock guard |
| Output | merged bf16 model → GCS |

**Why Base and not Instruct:** the Instruct checkpoint ships FP8-quantized, which does not
LoRA cleanly. Fara1.5 likewise SFT'd from a base (Qwen3.5-4B) checkpoint.

**Critical:** every image is resized to a fixed `1008×784`. Click targets in this dataset are
*normalized* `[0,1]`, so resizing preserves them — but the eval harness must apply the identical
resize or grounding will silently degrade.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime to A100"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} | {vram:.0f} GB | bf16={torch.cuda.is_bf16_supported()}")
if not torch.cuda.is_bf16_supported():
    print("WARNING: no bf16 (T4/V100). This notebook expects A100/L4.")

In [ ]:
%pip install -q -U "transformers>=5.0.0" "accelerate>=1.0" "peft>=0.14" \
    "datasets>=3.0" "huggingface_hub>=0.30" pillow
# Deliberately not building flash-attn: the compile can eat 15+ min of the budget.
# sdpa is used instead, and flash-attn is picked up automatically if preinstalled.

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# ---- Config ----
BASE_MODEL = "mistralai/Ministral-3-3B-Base-2512"

# Fixed image size. Both dims divisible by patch_size(14) AND by 14*2 for spatial_merge_size=2.
# 1008 = 72 patches, 784 = 56 patches -> merged 36x28 = 1008 vision tokens.
IMAGE_W, IMAGE_H = 1008, 784

# Sample mix. mind2web is on-distribution for Online-Mind2Web, so take all of it.
MIX = {
    "mind2web-l2.json":         {"zip": "mind2web.zip",           "take": None},
    "guiact-web-single.json":   {"zip": "guiact-web-single.zip",  "take": 6000},
    "miniwob-l2.json":          {"zip": "miniwob.zip",            "take": 1500},
}

EPOCHS = 1
MICRO_BATCH = 2
GRAD_ACCUM = 8            # effective batch 16
LR = 1e-4
LORA_R, LORA_ALPHA = 64, 128
TIME_BUDGET_HOURS = 3.0   # hard stop; model is still saved
MAX_TEXT_TOKENS = 2048    # truncation guard on the text side

OUT_DIR = "/content/ministral3-cua"
MERGED_DIR = "/content/Ministral3-3B-CUA-web"
GCS_DEST = "gs://ai-studio-bucket-347838016394-us-east1/usersim-models/Ministral3-3B-CUA-web"

import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---- Download the web subset (~2.5 GB) ----
import zipfile, json, time
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/aguvis"); DATA.mkdir(exist_ok=True)
IMG_ROOT = DATA / "images"; IMG_ROOT.mkdir(exist_ok=True)

t0 = time.time()
for js, spec in MIX.items():
    jp = hf_hub_download("xlangai/aguvis-stage2", js, repo_type="dataset")
    print(f"{js}: metadata ok")
    marker = IMG_ROOT / f".done_{spec['zip']}"
    if marker.exists():
        print(f"  {spec['zip']} already extracted"); continue
    zp = hf_hub_download("xlangai/aguvis-stage2", spec["zip"], repo_type="dataset")
    with zipfile.ZipFile(zp) as z:
        z.extractall(IMG_ROOT)
    marker.touch()
    print(f"  extracted {spec['zip']}")
print(f"download+extract took {(time.time()-t0)/60:.1f} min")

In [ ]:
# ---- Index every extracted image by basename (folder layouts differ per zip) ----
import collections

index = {}
for p in IMG_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        index.setdefault(p.name, p)
print(f"indexed {len(index)} images")

In [ ]:
# ---- Build flat training records ----
import random
random.seed(0)

def build(js, take):
    path = hf_hub_download("xlangai/aguvis-stage2", js, repo_type="dataset")
    raw = json.load(open(path))
    out, missing = [], 0
    for rec in raw:
        img = index.get(rec["image"]) if isinstance(rec["image"], str) else None
        if img is None:
            missing += 1; continue
        turns = rec["conversations"]
        system = next((t["value"] for t in turns if t["from"] == "system"), "")
        user = next((t["value"] for t in turns if t["from"] == "human"), "")
        gpt = [t["value"].strip() for t in turns if t["from"] == "gpt" and t["value"].strip()]
        if not user or not gpt:
            continue
        # Records carry a thought turn then an action turn; merge into one response.
        out.append({
            "image": str(img),
            "system": system,
            "user": user.replace("<image>", "[IMG]"),
            "assistant": "\n".join(gpt),
        })
    random.shuffle(out)
    if take:
        out = out[:take]
    print(f"{js}: {len(out)} records (missing images: {missing})")
    return out

records = []
for js, spec in MIX.items():
    records += build(js, spec["take"])
random.shuffle(records)

holdout, records = records[:200], records[200:]
print(f"\nTRAIN {len(records)} | HOLDOUT {len(holdout)}")
print("\n--- sample ---")
print(records[0]["user"][:300])
print("-> ", records[0]["assistant"][:200])

In [ ]:
# ---- Load model + processor ----
import torch
from transformers import AutoProcessor, Mistral3ForConditionalGeneration

processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.image_processor.size = {"longest_edge": max(IMAGE_W, IMAGE_H)}
tok = processor.tokenizer

attn = "sdpa"
try:
    import flash_attn  # noqa: F401
    attn = "flash_attention_2"
except Exception:
    pass
print("attention:", attn)

try:
    model = Mistral3ForConditionalGeneration.from_pretrained(
        BASE_MODEL, dtype=torch.bfloat16, attn_implementation=attn, device_map="cuda:0")
except TypeError:  # transformers < 5 keyword
    model = Mistral3ForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, attn_implementation=attn, device_map="cuda:0")

model.config.use_cache = False
print(model.config.model_type, f"{sum(p.numel() for p in model.parameters())/1e9:.2f}B params")

In [ ]:
# ---- Dataset ----
from PIL import Image
from torch.utils.data import Dataset

END_INST = tok.convert_tokens_to_ids("[/INST]")
print("[/INST] id:", END_INST)
EOS = tok.eos_token or "</s>"


def build_prompt(rec):
    sys_part = f"[SYSTEM_PROMPT]{rec['system']}[/SYSTEM_PROMPT]" if rec["system"] else ""
    return f"{sys_part}[INST]{rec['user']}[/INST]"


class CUADataset(Dataset):
    def __init__(self, recs):
        self.recs = recs

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        rec = self.recs[i]
        img = Image.open(rec["image"]).convert("RGB").resize((IMAGE_W, IMAGE_H), Image.BICUBIC)
        full = build_prompt(rec) + rec["assistant"] + EOS
        enc = processor(text=full, images=[img], return_tensors="pt")
        ids = enc["input_ids"][0]

        labels = ids.clone()
        # Mask the prompt: everything up to and including the final [/INST].
        pos = (ids == END_INST).nonzero()
        if len(pos):
            labels[: pos[-1].item() + 1] = -100
        else:  # fallback: re-encode the prompt alone to find the boundary
            n = processor(text=build_prompt(rec), images=[img],
                          return_tensors="pt")["input_ids"].shape[1]
            labels[:n] = -100

        item = {"input_ids": ids, "labels": labels,
                "attention_mask": torch.ones_like(ids)}
        for k in ("pixel_values", "image_sizes"):
            if k in enc:
                v = enc[k]
                item[k] = v[0] if (hasattr(v, "shape") and v.shape[0] == 1) else v
        return item


PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.unk_token_id or 0


def collate(batch):
    n = max(b["input_ids"].shape[0] for b in batch)
    n = min(n, MAX_TEXT_TOKENS + 4096)
    out = {}
    for key, fill in (("input_ids", PAD), ("labels", -100), ("attention_mask", 0)):
        rows = []
        for b in batch:
            t = b[key][:n]
            if t.shape[0] < n:
                t = torch.cat([t, torch.full((n - t.shape[0],), fill, dtype=t.dtype)])
            rows.append(t)
        out[key] = torch.stack(rows)
    for k in ("pixel_values", "image_sizes"):
        if k in batch[0]:
            try:
                out[k] = torch.stack([b[k] for b in batch])
            except Exception:
                out[k] = [b[k] for b in batch]
    return out


train_ds = CUADataset(records)
probe = train_ds[0]
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v)) for k, v in probe.items()})
print("supervised tokens:", int((probe["labels"] != -100).sum()), "/", probe["labels"].shape[0])
assert int((probe["labels"] != -100).sum()) > 0, "Label masking removed everything — check [/INST]"
print("decoded target:", tok.decode(probe["input_ids"][probe["labels"] != -100])[:300])

In [ ]:
# ---- LoRA: language model only; ViT frozen; multimodal projector trainable ----
from peft import LoraConfig, get_peft_model

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Restrict LoRA to the language tower so the vision encoder stays untouched.
lm_targets = sorted({
    n for n, _ in model.named_modules()
    if n.endswith(tuple(TARGETS)) and "vision_tower" not in n
})
print(f"{len(lm_targets)} LoRA target modules; e.g. {lm_targets[0]}")

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM", target_modules=lm_targets,
))

# The projector is small (~20M) and maps ViT features into the LM space — worth training.
proj = [m for n, m in model.named_modules() if n.endswith("multi_modal_projector")]
for m in proj:
    for p in m.parameters():
        p.requires_grad_(True)
print(f"unfroze {len(proj)} projector module(s)")

# Vision tower must stay frozen.
for n, p in model.named_parameters():
    if "vision_tower" in n:
        p.requires_grad_(False)

# Frozen embeddings + gradient checkpointing means no grad reaches the LoRA layers
# unless inputs are explicitly marked as requiring grad.
model.enable_input_require_grads()

model.print_trainable_parameters()

In [ ]:
# ---- Trainer with a hard wall-clock stop ----
import time
from transformers import Trainer, TrainingArguments, TrainerCallback


class TimeBudget(TrainerCallback):
    def __init__(self, hours):
        self.limit = hours * 3600
        self.t0 = None

    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()

    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > self.limit:
            print(f"\nTime budget {self.limit/3600:.1f}h reached — stopping, model still saves.")
            control.should_training_stop = True
        return control


args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    dataloader_num_workers=4,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Trainer(
    model=model, args=args, train_dataset=train_ds,
    data_collator=collate, callbacks=[TimeBudget(TIME_BUDGET_HOURS)],
)
print(f"optimizer steps: {len(train_ds)//(MICRO_BATCH*GRAD_ACCUM)*EPOCHS}")

In [ ]:
result = trainer.train()
print(result)
trainer.save_model(OUT_DIR)
processor.save_pretrained(OUT_DIR)

In [ ]:
# ---- Merge LoRA into the base weights -> single servable bf16 checkpoint ----
import gc

merged = model.merge_and_unload()
merged.config.use_cache = True
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
processor.save_pretrained(MERGED_DIR)
print("merged model written to", MERGED_DIR)
!du -sh {MERGED_DIR}

del trainer
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- Smoke test on a held-out sample ----
import re

rec = holdout[0]
img = Image.open(rec["image"]).convert("RGB").resize((IMAGE_W, IMAGE_H), Image.BICUBIC)
enc = processor(text=build_prompt(rec), images=[img], return_tensors="pt").to("cuda:0")

with torch.no_grad():
    out = merged.generate(**enc, max_new_tokens=128, do_sample=False)
pred = processor.tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print("TASK:", rec["user"][:200].replace("[IMG]", "").strip(), "\n")
print("PRED:", pred, "\n")
print("GOLD:", rec["assistant"])
print("\nparses as action:", bool(re.search(r"pyautogui\.\w+\(", pred)))

In [ ]:
# ---- Coordinate accuracy on the held-out split (cheap proxy for grounding) ----
def xy(text):
    m = re.search(r"x=([0-9.]+)\s*,\s*y=([0-9.]+)", text)
    return (float(m.group(1)), float(m.group(2))) if m else None

hits = tried = parsed = 0
for rec in holdout[:50]:
    gold = xy(rec["assistant"])
    if not gold:
        continue
    im = Image.open(rec["image"]).convert("RGB").resize((IMAGE_W, IMAGE_H), Image.BICUBIC)
    e = processor(text=build_prompt(rec), images=[im], return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        o = merged.generate(**e, max_new_tokens=128, do_sample=False)
    p = processor.tokenizer.decode(o[0][e["input_ids"].shape[1]:], skip_special_tokens=True)
    tried += 1
    pr = xy(p)
    if pr:
        parsed += 1
        # within 5% of the normalized canvas counts as a hit
        if abs(pr[0]-gold[0]) < 0.05 and abs(pr[1]-gold[1]) < 0.05:
            hits += 1

print(f"parsed {parsed}/{tried} | click within 5%: {hits}/{tried} = {hits/max(tried,1):.1%}")

In [ ]:
# ---- Ship it to GCS ----
from google.colab import auth
auth.authenticate_user()

!gcloud config set project project-amer-scs-sandbox
!gsutil -m cp -r {MERGED_DIR} {GCS_DEST}
!gsutil ls {GCS_DEST}
print("\nServe with:")
print(f"  gsutil -m cp -r {GCS_DEST} ./ && \\")
print("  vllm serve ./Ministral3-3B-CUA-web --dtype bfloat16 --max-model-len 16384 \\")
print("    --limit-mm-per-prompt image=5 --trust-remote-code")